# Loan Approval Prediction (Google Colab)

**Goal:** predict whether a loan application is approved (`Y`) or rejected (`N`) from applicant details.

**Steps**
1. Load and explore the data
2. EDA (target balance, categorical and numeric features, correlation)
3. Train/test split and preprocessing pipeline (imputation, scaling, one-hot encoding)
4. Baseline Logistic Regression and why accuracy alone is misleading
5. Handling class imbalance: class weights and **SMOTE**
6. Comparing several models with cross-validation
7. Hyperparameter tuning
8. Final evaluation (confusion matrix, ROC curve, feature importance)
9. Save the model and predict for a new applicant

Dataset: `loan.csv` (the class dataset). Run the cells from top to bottom.

## 1. Imports and loading the data
Upload `loan.csv` when asked (skipped automatically if the file is already in Colab).

In [ ]:
%pip install -q imbalanced-learn

import os, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")

CSV = "loan.csv"
if not os.path.exists(CSV):
    from google.colab import files
    up = files.upload()
    csvs = [n for n in up if n.lower().endswith(".csv")]
    if len(csvs) == 1 and csvs[0] != CSV:
        os.rename(csvs[0], CSV)

file = pd.read_csv(CSV)
print(file.shape)
file.head()

## 2. Explore the data

In [ ]:
file.info()
print("\nMissing values:\n", file.isna().sum())
print("\nDuplicate rows:", file.duplicated().sum())
file.describe().T

### EDA

In [ ]:
# Target balance
counts = file["Loan_Status"].value_counts()
print(counts, "\n", (counts / counts.sum() * 100).round(1).astype(str) + " %")
plt.figure(figsize=(4, 3))
sns.countplot(data=file, x="Loan_Status", palette=["#e45756", "#54a24b"])
plt.title("Loan approved (Y) vs rejected (N)"); plt.show()

In [ ]:
# Categorical features vs loan status
cat_cols = ["Gender", "Married", "Dependents", "Education", "Self_Employed", "Credit_History", "Property_Area"]
fig, axes = plt.subplots(2, 4, figsize=(16, 7))
for ax, col in zip(axes.ravel(), cat_cols):
    sns.countplot(data=file, x=col, hue="Loan_Status", ax=ax, palette=["#e45756", "#54a24b"])
    ax.set_title(col)
axes.ravel()[-1].axis("off")
plt.tight_layout(); plt.show()

In [ ]:
# Numeric features
num_cols = ["ApplicantIncome", "CoapplicantIncome", "LoanAmount", "Loan_Amount_Term"]
fig, axes = plt.subplots(2, 4, figsize=(16, 6))
for i, col in enumerate(num_cols):
    sns.histplot(file[col].dropna(), kde=True, ax=axes[0, i], color="#4c78a8")
    sns.boxplot(data=file, x="Loan_Status", y=col, ax=axes[1, i], palette=["#e45756", "#54a24b"])
    axes[0, i].set_title(col)
plt.tight_layout(); plt.show()

# Correlation (numeric + target)
tmp = file[num_cols + ["Credit_History"]].copy()
tmp["Loan_Status"] = file["Loan_Status"].map({"N": 0, "Y": 1})
plt.figure(figsize=(6, 4))
sns.heatmap(tmp.corr(), annot=True, fmt=".2f", cmap="coolwarm")
plt.title("Correlation"); plt.show()

**What we see:** about 70% of applications are approved, so the data is **imbalanced**.
Check the plots above: `Credit_History` usually shows the clearest difference between approved and rejected loans, and income columns are right-skewed with outliers.

## 3. Features, target, train/test split

In [ ]:
X = file.drop(columns=["Loan_Status", "Loan_ID"])
y = file["Loan_Status"].map({"N": 0, "Y": 1})       # 1 = approved, 0 = rejected

from sklearn.model_selection import train_test_split
x_train, x_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42)     # stratify keeps the Y/N ratio equal in both parts

print("Train:", x_train.shape, " Test:", x_test.shape)
print("Train class ratio:\n", y_train.value_counts(normalize=True).round(3))

## 4. Preprocessing pipeline
- Numeric columns: fill missing with the **median**, then **StandardScaler**
- Categorical columns: fill missing with the **most frequent** value, then **OneHotEncoder**

Everything is learned from the training data only (inside a pipeline), so nothing leaks from the test set.

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

numeric = ["ApplicantIncome", "CoapplicantIncome", "LoanAmount", "Loan_Amount_Term"]
categorical = ["Gender", "Married", "Dependents", "Education", "Self_Employed", "Credit_History", "Property_Area"]

num_pipe = Pipeline([("impute", SimpleImputer(strategy="median")),
                     ("scale", StandardScaler())])
cat_pipe = Pipeline([("impute", SimpleImputer(strategy="most_frequent")),
                     ("encode", OneHotEncoder(handle_unknown="ignore", sparse_output=False))])

preprocess = ColumnTransformer([("num", num_pipe, numeric),
                                ("cat", cat_pipe, categorical)])
preprocess

## 5. Baseline model: Logistic Regression

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, classification_report, confusion_matrix,
                             ConfusionMatrixDisplay, recall_score, f1_score, roc_auc_score, roc_curve)

baseline = Pipeline([("prep", preprocess), ("model", LogisticRegression(max_iter=1000))])
baseline.fit(x_train, y_train)

pred_train = baseline.predict(x_train)
pred_test = baseline.predict(x_test)
print("Train accuracy:", round(accuracy_score(y_train, pred_train), 4))
print("Test accuracy :", round(accuracy_score(y_test, pred_test), 4))
print("\nTest report:\n", classification_report(y_test, pred_test, target_names=["Rejected (0)", "Approved (1)"]))

ConfusionMatrixDisplay(confusion_matrix(y_test, pred_test), display_labels=["Rejected", "Approved"]).plot(cmap="Blues")
plt.title("Baseline: confusion matrix (test)"); plt.show()

**Problem:** the model looks fine on accuracy, but it is much better at spotting *approved* loans than *rejected* ones
(low recall for class 0). A model that says "approve everyone" already gets ~70% accuracy,
so for imbalanced data we also look at **recall of the rejected class** and **macro F1**.

## 6. Handling class imbalance

In [ ]:
from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.over_sampling import SMOTE

# Option A: class weights (the model pays more attention to the minority class)
weighted = Pipeline([("prep", preprocess),
                     ("model", LogisticRegression(max_iter=1000, class_weight="balanced"))])

# Option B: SMOTE (creates synthetic minority samples). It is applied ONLY to training data inside the pipeline.
smote = ImbPipeline([("prep", preprocess),
                     ("smote", SMOTE(random_state=42)),
                     ("model", LogisticRegression(max_iter=1000))])

def evaluate(name, pipe):
    pipe.fit(x_train, y_train)
    p = pipe.predict(x_test)
    return {"Model": name,
            "Accuracy": accuracy_score(y_test, p),
            "Recall (rejected)": recall_score(y_test, p, pos_label=0),
            "Recall (approved)": recall_score(y_test, p, pos_label=1),
            "Macro F1": f1_score(y_test, p, average="macro")}

res = pd.DataFrame([evaluate("Baseline LR", baseline),
                    evaluate("LR + class_weight", weighted),
                    evaluate("LR + SMOTE", smote)]).round(3)
print(res.to_string(index=False))

In [ ]:
# Visual check of what SMOTE does to the training set size
prep_only = preprocess.fit_transform(x_train)
Xs, ys = SMOTE(random_state=42).fit_resample(prep_only, y_train)
fig, ax = plt.subplots(1, 2, figsize=(8, 3))
y_train.value_counts().sort_index().plot(kind="bar", ax=ax[0], color="#4c78a8", title="Before SMOTE")
pd.Series(ys).value_counts().sort_index().plot(kind="bar", ax=ax[1], color="#54a24b", title="After SMOTE")
plt.tight_layout(); plt.show()

## 7. Compare several models (5-fold cross-validation on the training set)

In [ ]:
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.metrics import make_scorer
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC

models = {
    "Logistic Regression": LogisticRegression(max_iter=1000),
    "Decision Tree": DecisionTreeClassifier(max_depth=5, random_state=42),
    "Random Forest": RandomForestClassifier(n_estimators=200, random_state=42),
    "Gradient Boosting": GradientBoostingClassifier(random_state=42),
    "KNN": KNeighborsClassifier(n_neighbors=7),
    "SVM": SVC(probability=True, random_state=42),
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scoring = {"accuracy": "accuracy",
           "macro_f1": "f1_macro",
           "recall_rejected": make_scorer(recall_score, pos_label=0)}

rows = []
for name, clf in models.items():
    pipe = ImbPipeline([("prep", preprocess), ("smote", SMOTE(random_state=42)), ("model", clf)])
    s = cross_validate(pipe, x_train, y_train, cv=cv, scoring=scoring)
    rows.append({"Model": name,
                 "CV Accuracy": s["test_accuracy"].mean(),
                 "CV Macro F1": s["test_macro_f1"].mean(),
                 "CV Recall (rejected)": s["test_recall_rejected"].mean()})

cv_results = pd.DataFrame(rows).sort_values("CV Macro F1", ascending=False).round(3)
print(cv_results.to_string(index=False))

cv_results.set_index("Model")[["CV Accuracy", "CV Macro F1", "CV Recall (rejected)"]].plot(
    kind="bar", figsize=(9, 4), rot=30, title="Model comparison (SMOTE, 5-fold CV)")
plt.ylim(0, 1); plt.show()

## 8. Hyperparameter tuning
We tune Logistic Regression and Random Forest with `GridSearchCV` and keep the better one.

In [ ]:
from sklearn.model_selection import GridSearchCV

def make_pipe(clf):
    return ImbPipeline([("prep", preprocess), ("smote", SMOTE(random_state=42)), ("model", clf)])

searches = {
    "Logistic Regression": GridSearchCV(
        make_pipe(LogisticRegression(max_iter=2000)),
        {"model__C": [0.01, 0.1, 1, 10, 100]},
        cv=cv, scoring="f1_macro", n_jobs=-1),
    "Random Forest": GridSearchCV(
        make_pipe(RandomForestClassifier(random_state=42)),
        {"model__n_estimators": [100, 300],
         "model__max_depth": [3, 5, 10, None],
         "model__min_samples_leaf": [1, 3, 5]},
        cv=cv, scoring="f1_macro", n_jobs=-1),
}

for name, gs in searches.items():
    gs.fit(x_train, y_train)
    print(f"{name:<20} best CV macro F1 = {gs.best_score_:.3f}   params = {gs.best_params_}")

best_name = max(searches, key=lambda k: searches[k].best_score_)
final_model = searches[best_name].best_estimator_
print("\nSelected model:", best_name)

## 9. Final evaluation on the untouched test set

In [ ]:
pred = final_model.predict(x_test)
proba = final_model.predict_proba(x_test)[:, 1]

print("Test accuracy:", round(accuracy_score(y_test, pred), 4))
print("ROC AUC      :", round(roc_auc_score(y_test, proba), 4))
print("\n", classification_report(y_test, pred, target_names=["Rejected (0)", "Approved (1)"]))

fig, ax = plt.subplots(1, 2, figsize=(11, 4))
ConfusionMatrixDisplay(confusion_matrix(y_test, pred), display_labels=["Rejected", "Approved"]).plot(ax=ax[0], cmap="Blues", colorbar=False)
ax[0].set_title("Confusion matrix (test)")
fpr, tpr, _ = roc_curve(y_test, proba)
ax[1].plot(fpr, tpr, label=f"AUC = {roc_auc_score(y_test, proba):.2f}")
ax[1].plot([0, 1], [0, 1], "--", color="gray")
ax[1].set_xlabel("False positive rate"); ax[1].set_ylabel("True positive rate"); ax[1].set_title("ROC curve"); ax[1].legend()
plt.tight_layout(); plt.show()

# compare with the original baseline
print("\nBaseline test accuracy:", round(accuracy_score(y_test, baseline.predict(x_test)), 4),
      "| Baseline recall (rejected):", round(recall_score(y_test, baseline.predict(x_test), pos_label=0), 3))
print("Final    test accuracy:", round(accuracy_score(y_test, pred), 4),
      "| Final    recall (rejected):", round(recall_score(y_test, pred, pos_label=0), 3))

### Which features matter most?

In [ ]:
names = final_model.named_steps["prep"].get_feature_names_out()
est = final_model.named_steps["model"]
if hasattr(est, "feature_importances_"):
    imp = pd.Series(est.feature_importances_, index=names)
else:
    imp = pd.Series(est.coef_[0], index=names)

imp = imp.reindex(imp.abs().sort_values(ascending=False).index).head(12)
imp.sort_values().plot(kind="barh", figsize=(7, 5), color="#4c78a8", title="Top features")
plt.show()

## 10. Save the model and predict for a new applicant

In [ ]:
import joblib
joblib.dump(final_model, "loan_model.joblib")
print("Saved loan_model.joblib")

def predict_loan(applicant: dict):
    df = pd.DataFrame([applicant])
    p_approve = final_model.predict_proba(df)[0, 1]
    decision = "APPROVED" if p_approve >= 0.5 else "REJECTED"
    return decision, p_approve

new_applicant = {
    "Gender": "Male", "Married": "Yes", "Dependents": "0", "Education": "Graduate",
    "Self_Employed": "No", "ApplicantIncome": 5000, "CoapplicantIncome": 1500,
    "LoanAmount": 120, "Loan_Amount_Term": 360, "Credit_History": 1.0, "Property_Area": "Urban",
}
decision, prob = predict_loan(new_applicant)
print(f"Decision: {decision} (probability of approval = {prob:.2%})")

bad_applicant = dict(new_applicant, Credit_History=0.0, ApplicantIncome=2000, CoapplicantIncome=0, LoanAmount=200)
decision, prob = predict_loan(bad_applicant)
print(f"Decision: {decision} (probability of approval = {prob:.2%})")

try:
    from google.colab import files
    files.download("loan_model.joblib")
except Exception:
    pass

## Conclusion
- The data is imbalanced (about 70% approved), so **accuracy alone is misleading**; we also track recall of rejected loans and macro F1.
- The class baseline misses many rejected applications (low recall for class 0). Compare it with the **class_weight** and **SMOTE** results in the tables above to see whether recall of rejected loans improved, and at what cost in accuracy.
- Splitting the data **before** any resampling or fitting avoids data leakage (the class notebook evaluated on data the model had already seen).
- See the feature importance plot for the most influential features (for this kind of data `Credit_History` is usually on top).
- The final model is saved as `loan_model.joblib` and can predict for new applicants with `predict_loan(...)`.

**Limitations:** the dataset is small (381 rows), so results change a little with a different random split.